# 02 - Train LSTM Load Forecaster

**ITAI 2376 Final Project - GridGuard-AI**  
**Authors:** DeMarcus Crump & Yoana Cook

Trains a **2-layer stacked LSTM** that forecasts ERCOT load one hour ahead from a 24-hour input sequence. At runtime the Grid Monitor agent compares the actual load to this model's prediction; a deviation > 5 % flags a physical grid anomaly.

### Blueprint-locked specification
- **Architecture:** `LSTM(64) -> LSTM(32) -> Dense(16) -> Dense(1)`  (~29 k trainable parameters)
- **Sequence length:** 24 hours
- **Training window:** last 90 days of `gridguard.db`
- **Target metric:** <= 2.5 % MAE of peak load on a held-out tail split
- **Output artifact:** `models/load_forecaster.h5`  +  `models/load_forecaster_scaler.npz`  +  `models/load_forecaster_config.json`

### Deep-learning connection (Module 04 - Recurrent Neural Networks)
Grid balancing is an inherently sequential problem: the load at hour *t* depends on loads at *t-1*, *t-2*, ... , *t-24*. LSTM cells use gated memory to retain that 24-hour context while forgetting irrelevant noise - the same architecture pattern we studied in Module 04.

Expected runtime in Colab (CPU): ~3 min. (GPU): ~1 min.

In [ ]:
# --- Colab bootstrap ---
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install pandas scikit-learn matplotlib tensorflow==2.16.1
    REPO_URL = os.environ.get('GRIDGUARD_REPO_URL', 'https://github.com/<YOUR_GITHUB_USER>/Crump_GridGuard_ITAI2376.git')
    if not os.path.isdir('/content/repo'):
        !git clone {REPO_URL} /content/repo
    os.chdir('/content/repo')
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
print('Working directory:', os.getcwd())

In [ ]:
import os, json, sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

np.random.seed(42)
tf.random.set_seed(42)
print('TensorFlow:', tf.__version__)

In [ ]:
# --- Load last 90 days from gridguard.db ---
SEQUENCE_LENGTH = 24        # 24-hour input window (blueprint-locked)
TRAINING_DAYS   = 90        # 90-day training window (blueprint-locked)
TARGET_COL      = 'load_mw'

# Auxiliary features fed alongside load. Kept small on purpose so the model
# learns the temporal dynamics rather than memorising exogenous signals.
FEATURE_COLS = [
    'load_mw',
    'temp_avg_statewide',
    'wind_avg_statewide',
    'humidity_avg_statewide',
    'hour_sin', 'hour_cos',
    'is_weekend',
]

conn = sqlite3.connect('data/gridguard.db')
df = pd.read_sql(
    f"SELECT timestamp, {', '.join(FEATURE_COLS)} FROM ercot_telemetry ORDER BY timestamp",
    conn, parse_dates=['timestamp']
).dropna()
conn.close()

cutoff = df['timestamp'].max() - pd.Timedelta(days=TRAINING_DAYS)
df = df[df['timestamp'] >= cutoff].reset_index(drop=True)
print(f'Rows in 90-day window: {len(df):,}')
print(f'Date range           : {df["timestamp"].min()} -> {df["timestamp"].max()}')
df.head()

In [ ]:
# --- Scale features, build sliding windows ---
values = df[FEATURE_COLS].values.astype('float32')
scaler = MinMaxScaler()
scaled = scaler.fit_transform(values)

target_idx = FEATURE_COLS.index(TARGET_COL)

def make_sequences(data, seq_len, target_idx):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len, target_idx])
    return np.asarray(X), np.asarray(y)

X, y = make_sequences(scaled, SEQUENCE_LENGTH, target_idx)
print(f'X shape: {X.shape}  (samples, seq_len, features)')
print(f'y shape: {y.shape}')

# 80 / 20 temporal split (the last 20 % is the held-out test tail)
split = int(len(X) * 0.80)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
print(f'Train: {len(X_train):,}   Test: {len(X_test):,}')

In [ ]:
# --- Build the 2-layer stacked LSTM (~29 k parameters) ---
n_features = X.shape[2]

model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(SEQUENCE_LENGTH, n_features)),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])
model.compile(optimizer='adam', loss='huber', metrics=['mae'])
model.summary()

In [ ]:
# --- Train with early stopping ---
callbacks = [
    EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss'),
    ReduceLROnPlateau(factor=0.5, patience=4, monitor='val_loss'),
]

history = model.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=60,
    batch_size=32,
    callbacks=callbacks,
    verbose=2,
)

# --- Training curves ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train'); axes[0].plot(history.history['val_loss'], label='val'); axes[0].set_title('Huber loss'); axes[0].legend()
axes[1].plot(history.history['mae'], label='train'); axes[1].plot(history.history['val_mae'], label='val'); axes[1].set_title('MAE (scaled)'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# --- Evaluate on the held-out tail ---
y_pred_scaled = model.predict(X_test, verbose=0).ravel()

# Inverse-scale only the load column
def inverse_load(values_scaled):
    pad = np.zeros((len(values_scaled), n_features))
    pad[:, target_idx] = values_scaled
    return scaler.inverse_transform(pad)[:, target_idx]

y_test_mw  = inverse_load(y_test)
y_pred_mw  = inverse_load(y_pred_scaled)

peak_load = df[TARGET_COL].max()
mae_mw    = mean_absolute_error(y_test_mw, y_pred_mw)
rmse_mw   = float(np.sqrt(mean_squared_error(y_test_mw, y_pred_mw)))
mape_pct  = float(np.mean(np.abs((y_test_mw - y_pred_mw) / y_test_mw)) * 100)
mae_pct   = float(mae_mw / peak_load * 100)

print(f'Test MAE      : {mae_mw:,.0f} MW')
print(f'Test RMSE     : {rmse_mw:,.0f} MW')
print(f'Test MAPE     : {mape_pct:.2f} %')
print(f'MAE % of peak : {mae_pct:.2f} %   (target <= 2.5 %)')

# --- Visualize the first 200 hours of predictions vs ground truth ---
plt.figure(figsize=(12, 4))
plt.plot(y_test_mw[:200], label='Actual', linewidth=1.2)
plt.plot(y_pred_mw[:200], label='LSTM prediction', linewidth=1.2, alpha=0.8)
plt.ylabel('Load (MW)'); plt.xlabel('Hour (held-out test tail)'); plt.title('LSTM load forecast vs. actual')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# --- Persist artifacts that the Grid Monitor agent will load at runtime ---
os.makedirs('models', exist_ok=True)

H5_PATH     = 'models/load_forecaster.h5'
SCALER_PATH = 'models/load_forecaster_scaler.npz'
CONFIG_PATH = 'models/load_forecaster_config.json'

model.save(H5_PATH)
np.savez(SCALER_PATH, data_min_=scaler.data_min_, data_max_=scaler.data_max_, scale_=scaler.scale_, min_=scaler.min_)

config = {
    'trained_at'       : pd.Timestamp.utcnow().isoformat(),
    'sequence_length'  : SEQUENCE_LENGTH,
    'training_days'    : TRAINING_DAYS,
    'features'         : FEATURE_COLS,
    'target'           : TARGET_COL,
    'architecture'     : 'LSTM(64) -> LSTM(32) -> Dense(16) -> Dense(1)',
    'trainable_params' : int(model.count_params()),
    'anomaly_threshold_pct': 5.0,
    'peak_load_mw'     : float(peak_load),
    'test_metrics'     : {
        'mae_mw'        : float(mae_mw),
        'rmse_mw'       : rmse_mw,
        'mape_pct'      : mape_pct,
        'mae_pct_of_peak': mae_pct,
    },
}
with open(CONFIG_PATH, 'w') as fh:
    json.dump(config, fh, indent=2)

print('Saved:')
print('  ', H5_PATH)
print('  ', SCALER_PATH)
print('  ', CONFIG_PATH)
print('\nTrainable params:', model.count_params())

In [ ]:
# --- Colab convenience: download the artifacts back to your machine ---
if IN_COLAB:
    from google.colab import files
    for path in [H5_PATH, SCALER_PATH, CONFIG_PATH]:
        files.download(path)
    print('Downloaded. Drop these three files into the academic repo models/ folder and commit.')
else:
    print('Artifacts already written to models/ locally. Commit them with git.')